# Реализация моделей NER

Этот ноутбук содержит реализацию трех различных моделей для задачи распознавания именованных сущностей (NER): классический метод (CRF), нейросетевой метод (Bi-LSTM-CRF) и метод на основе трансформеров (BERT fine-tuning). Данные для обучения и оценки моделей предполагаются подготовленными в предыдущем ноутбуке (`ner_preprocessing_and_models.ipynb`).

## 1. Классический метод (CRF)

Для CRF модели необходимо извлечь признаки из предложений. Эти признаки будут использоваться библиотекой `sklearn-crfsuite`. Признаки включают в себя информацию о самом слове, его регистре, наличии цифр, а также о частях речи (POS-тегах) текущего, предыдущего и следующего слов. Для получения POS-тегов используется библиотека `nltk`.

In [ ]:
import sklearn_crfsuite
import nltk
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')

def word2features(sent, i):
    """
    Извлекает признаки для слова в предложении.
    :param sent: Словарь, содержащий 'tokens' (список слов) и 'ner_tags' (список NER-тегов).
    :param i: Индекс текущего слова в предложении.
    :return: Словарь признаков для текущего слова.
    """
    word = sent['tokens'][i]
    postag = nltk.pos_tag([word])[0][1]

    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word.isupper()': word.isupper(),
        'word.istitle()': word.istitle(),
        'word.isdigit()': word.isdigit(),
        'postag': postag,
        'postag[:2]': postag[:2],
    }
    if i > 0:
        word1 = sent['tokens'][i-1]
        postag1 = nltk.pos_tag([word1])[0][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True  # Начало предложения

    if i < len(sent['tokens'])-1:
        word1 = sent['tokens'][i+1]
        postag1 = nltk.pos_tag([word1])[0][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True  # Конец предложения

    return features

def sent2features(sent):
    """
    Извлекает признаки для всех слов в предложении.
    """
    return [word2features(sent, i) for i in range(len(sent['tokens']))]

def sent2labels(sent):
    """
    Возвращает NER-метки для всех слов в предложении.
    """
    return [tag for tag in sent['ner_tags']]

def sent2tokens(sent):
    """
    Возвращает токены для всех слов в предложении.
    """
    return [token for token in sent['tokens']]


### 1.1. Подготовка данных для CRF

Предполагается, что `train_data`, `val_data`, `test_data` уже определены и имеют структуру, ожидаемую функциями `sent2features` и `sent2labels` (например: `[{'tokens': ['Word1', 'Word2'], 'ner_tags': ['O', 'B-PER']}]`). Эти данные должны быть загружены из `ner_preprocessing_and_models.ipynb`.

In [ ]:
# Пример заглушек для данных, если они не определены в предыдущих ячейках
# train_data = [{'tokens': ['Barack', 'Obama', 'was', 'born', 'in', 'Hawaii', '.'], 'ner_tags': ['B-PER', 'I-PER', 'O', 'O', 'O', 'B-LOC', 'O']}]
# val_data = [{'tokens': ['He', 'lives', 'in', 'Washington', 'D.C.', '.'], 'ner_tags': ['O', 'O', 'O', 'B-LOC', 'I-LOC', 'O']}]
# test_data = [{'tokens': ['Apple', 'is', 'a', 'company', '.'], 'ner_tags': ['B-ORG', 'O', 'O', 'O', 'O']}]

# Для реального использования, убедитесь, что train_data, val_data, test_data доступны из предыдущего ноутбука.
# Если вы запускаете этот ноутбук отдельно, вам нужно будет загрузить или определить эти переменные.

X_train_crf = [sent2features(s) for s in train_data]
y_train_crf = [sent2labels(s) for s in train_data]

X_val_crf = [sent2features(s) for s in val_data]
y_val_crf = [sent2labels(s) for s in val_data]

X_test_crf = [sent2features(s) for s in test_data]
y_test_crf = [sent2labels(s) for s in test_data]

print("Пример признаков для первого слова первого предложения:")
print(X_train_crf[0][0])
print("\nПример меток для первого предложения:")
print(y_train_crf[0])


### 1.2. Инициализация и обучение CRF модели

Инициализируем модель `sklearn_crfsuite.CRF` с заданными параметрами. После инициализации модель может быть обучена на подготовленных данных.

In [ ]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
print("\nCRF модель инициализирована.")
# Для обучения модели:
# crf.fit(X_train_crf, y_train_crf)


## 2. Нейросетевой метод (Bi-LSTM-CRF)

Реализуем архитектуру Bi-LSTM-CRF с использованием TensorFlow/Keras. Модель будет принимать на вход последовательности токенов (или их эмбеддинги) и выдавать последовательности тегов NER. Архитектура включает слой эмбеддингов, Bi-LSTM слои и CRF слой. Для CRF слоя в Keras будем использовать библиотеку `tensorflow_addons`.

In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import tensorflow_addons as tfa

def build_bilstm_crf_model(max_seq_length, num_tags, embedding_dim=100, lstm_units=256):
    """
    Строит модель Bi-LSTM-CRF для задачи NER.
    :param max_seq_length: Максимальная длина входной последовательности.
    :param num_tags: Количество уникальных NER-тегов.
    :param embedding_dim: Размерность эмбеддингов слов.
    :param lstm_units: Количество юнитов в LSTM слоях.
    :return: Скомпилированная модель Keras.
    """
    word_inputs = layers.Input(shape=(max_seq_length,), dtype=tf.int32, name="word_inputs")

    # Слой эмбеддингов. Предполагаем, что у нас есть словарь токенов.
    # num_words - это размер словаря токенов. Для CoNLL-2003 можно оценить размер словаря из train_data.
    # В данном случае, мы не имеем предопределенного словаря, поэтому используем placeholder.
    # Для корректной работы, нужно будет создать слой StringLookup или TextVectorization
    # для маппинга токенов в ID.
    num_words = 50000  # Примерное количество уникальных слов в словаре
    embedding_layer = layers.Embedding(input_dim=num_words, output_dim=embedding_dim, mask_zero=True)(word_inputs)

    # Bi-LSTM слои
    bilstm = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=True))(embedding_layer)
    bilstm = layers.Dropout(0.3)(bilstm)  # Добавляем Dropout для регуляризации

    # Проекция на пространство тегов
    dense = layers.TimeDistributed(layers.Dense(num_tags))(bilstm)

    # CRF слой
    crf_layer = tfa.layers.CRF(num_tags, name="crf_layer")
    output = crf_layer(dense)

    model = Model(inputs=word_inputs, outputs=output)

    # Компиляция модели с CRF loss
    model.compile(optimizer=Adam(learning_rate=0.001), loss=crf_layer.loss, metrics=[crf_layer.accuracy])

    return model


### 2.1. Инициализация Bi-LSTM-CRF модели

Для инициализации модели необходимы `MAX_SEQ_LENGTH` (максимальная длина последовательности) и `num_tags` (количество уникальных тегов). Эти переменные должны быть определены в предыдущих ячейках или загружены из `ner_preprocessing_and_models.ipynb`.

In [ ]:
# MAX_SEQ_LENGTH и tag_to_id должны быть определены в предыдущих ячейках или здесь.
# Пример заглушек:
# MAX_SEQ_LENGTH = 128
# tag_to_id = {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-LOC': 3, 'I-LOC': 4, 'B-ORG': 5, 'I-ORG': 6}
# num_tags = len(tag_to_id)

bilstm_crf_model = build_bilstm_crf_model(MAX_SEQ_LENGTH, num_tags)
bilstm_crf_model.summary()

print("\nBi-LSTM-CRF модель инициализирована.")


## 3. Метод на основе трансформеров (BERT/RoBERTa fine-tuning)

Реализуем модель для fine-tuning предобученной модели BERT для задачи NER. Модель будет использовать `TFBertForTokenClassification` из библиотеки `transformers`, которая уже имеет классификационный слой поверх BERT и предназначена для задач токенизации, таких как NER.

In [ ]:
from transformers import TFBertForTokenClassification, AutoTokenizer, TFAutoModelForTokenClassification
import tensorflow as tf

def build_bert_ner_model(bert_model_name, num_tags):
    """
    Строит модель BERT для задачи NER.
    :param bert_model_name: Имя предобученной модели BERT (например, 'bert-base-uncased').
    :param num_tags: Количество уникальных NER-тегов.
    :return: Скомпилированная модель Keras на основе BERT.
    """
    # TFBertForTokenClassification уже имеет классификационный слой поверх BERT
    # и предназначен для задач токенизации, таких как NER.
    # Он автоматически добавляет слой классификации с num_labels выходами.
    model = TFBertForTokenClassification.from_pretrained(bert_model_name, num_labels=num_tags)

    # Компиляция модели
    # Важно использовать SparseCategoricalCrossentropy, так как метки не one-hot encoded
    # from_logits=True, потому что выходной слой BERT не применяет softmax
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)  # Типичный learning rate для fine-tuning BERT

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=['accuracy'])

    return model


### 3.1. Инициализация BERT NER модели

Для инициализации модели BERT NER необходимо указать имя предобученной модели (например, `bert-base-uncased`) и количество уникальных тегов (`num_tags`). `tag_to_id` и `MAX_SEQ_LENGTH` должны быть доступны из предыдущего ноутбука.

In [ ]:
# bert_model_name и num_tags должны быть определены.
# Пример заглушек:
# model_name = "bert-base-uncased"
# num_tags = len(tag_to_id)  # tag_to_id должен быть определен

bert_ner_model = build_bert_ner_model(model_name, num_tags)
bert_ner_model.summary()

print("\nBERT NER модель инициализирована.")
